# Israeli legal eval with the corpus, part 2 of 2: DictaLM 3.0 24B Thinking

Answers **100 questions** (the stratified sample, seed 2026) and the **first 5 paralegal cases**, with
retrieval over the bulk legal corpus, with **DictaLM 3.0 24B Thinking**
(`dicta-il/DictaLM-3.0-24B-Thinking-GGUF`, pulled from Hugging Face -- it always reasons). **No judging here.**

**Time**: this is the slow one. Without the corpus it took ~7.5 hours (it reasons at length and often runs
to its 6,144-token limit); with the corpus the prompts are longer, so expect **7-9 hours** -- inside
Kaggle's 12-hour session, so start it on its own, as a committed run.

**Part of a pair**: `kaggle_eval_qwen14b_gemma12b.ipynb` and `kaggle_eval_dictalm24b.ipynb` split the
three-model comparison into two Kaggle sessions, so each finishes well inside the 12-hour session cap and
together they fit the weekly GPU quota. Both use exactly the same settings -- the same 100 questions
(seed 2026), 5 cases, corpus, context and output limits -- so their results compare directly. **Don't
change the settings between the two runs.**

What the run leaves (under `/kaggle/working/AI-IZ/data/legal/eval_qwen_gemma_dictalm/<RUN_NAME>/<model>/`):
- `bulk500/answers_rag.jsonl` -- each answer, with the retrieval plan, the retrieved provisions and repairs;
- `cases/answers_rag.jsonl` -- the 5 work files;
- `llm_trace/*.jsonl` -- **every model call in full**: prompt, reasoning, output, stop reason, tokens, timing;
- `reasoning_report.md` and `reasoning_report_with_prompts.md`;
- `judge_free_report.txt/json` -- multiple choice and yes/no labels, scored without a judge.

Plus `summary.md/json`, `corpus_coverage.md/json`, `environment.txt`, `ollama.log`, and
**`<RUN_NAME>_results.zip`** in Output -- send that zip back for the judging and the reasoning review.

## Before running
1. *Add-ons -> Secrets*: **GITHUB_TOKEN** (reads `zananiri/AI-IZ`).
2. *Add Input*: the dataset with **`legal_corpus_vectordb`** (zipped or unpacked). The corpus cell finds the
   database anywhere in the input and **stops with an error if there is none** -- no run without the corpus.
3. *Settings -> Accelerator*: **GPU T4 x2**. *Settings -> Internet*: **on**.
4. *Save Version -> Save & Run All (Commit)*, so it runs in the background and the zip lands in Output.
   After the corpus cell, the log shows the chunk counts and `corpus OK: ['laws', 'procedural_rules']`.

## If the session ends early
Attach this notebook's earlier version (its Output holds the results zip) as Input, set `RESUME_FROM = "auto"`,
keep `RUN_NAME` and everything else unchanged, and run again: finished questions are skipped, answers that
errored or came back empty are redone.


In [ ]:
BRANCH = "main"            # the branch to test
MODELS = [
    {"repo": "dicta-il/DictaLM-3.0-24B-Thinking-GGUF", "quant": None,  # None = the best 4-5 bit GGUF file
     "thinking": True},                                              # the repo has (checked at run time)
]
LIMIT = 100             # questions: a stratified sample across the 9 categories; None = all 500
SAMPLE_SEED = 2026      # the same seed as the other runs -> the same 100 questions
RUN_CASES = True        # the paralegal cases
CASE_LIMIT = 5          # the first N cases of cases_gold.json
TOP_K = 12              # retrieved chunks per question (issue spotting: 16, set in eval_run.py)
CASE_TOP_K = 16         # retrieved chunks per case
CONTEXT_LENGTH = 16384  # tokens per call (Ollama num_ctx): prompt + reasoning + answer
ANSWER_MAX_TOKENS = 6144
CASE_MAX_TOKENS = 6144
REQUEST_TIMEOUT_S = 1800  # per model call; a long thinking answer on a T4 can take many minutes

RUN_NAME = "corpus_dictalm24b"   # RUN_DIR (data/legal/eval_qwen_gemma_dictalm/RUN_NAME) is set in step 1, after os.chdir(REPO)
RESUME_FROM = None      # a path (or "auto") to an earlier run's results zip/folder attached as Input; None = fresh


## 1. Code and packages

In [ ]:
import os, pathlib, shutil, subprocess, json, time, urllib.request
from kaggle_secrets import UserSecretsClient

def sh(command, log=None, check=True, env=None):
    """Runs a shell command with its output streamed live (and appended to `log`); raises on
    failure when check=True -- unlike `!command`, which carries on silently."""
    proc = subprocess.Popen(["bash", "-o", "pipefail", "-c", command], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    out = open(log, "a", encoding="utf-8") if log else None
    for line in proc.stdout:
        print(line, end="", flush=True)
        if out:
            out.write(line)
    if out:
        out.close()
    if proc.wait() and check:
        raise RuntimeError(f"exit code {proc.returncode}: {command}")
    return proc.returncode

sh("nvidia-smi --query-gpu=name,memory.total --format=csv")

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO = "/kaggle/working/AI-IZ"
if not pathlib.Path(REPO, ".git").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                    f"https://{token}@github.com/zananiri/AI-IZ.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", "https://github.com/zananiri/AI-IZ.git"], check=True)
os.chdir(REPO)
RUN_DIR = pathlib.Path(f"data/legal/eval_qwen_gemma_dictalm/{RUN_NAME}")
RUN_DIR.mkdir(parents=True, exist_ok=True)
sh("git log --oneline -1")

sh('pip install -q uv')
sh(f'uv pip install -q --system -e ".[legal,legal-data]"')


## 2. Ollama and DictaLM

`OLLAMA_MAX_LOADED_MODELS=1`: one model on the GPU at a time. Flash attention + an 8-bit KV cache halve the 16k context's memory. Retrieval (bge-m3 + the reranker) runs on the CPU in step 6, so the GPU stays with the model.

DictaLM isn't an Ollama library model, so it has no fixed `tag` up front -- it's pulled from its Hugging Face GGUF repo, picking the first quantization in `PREFERRED_QUANTS` the repo actually has (or `quant` if a model spec sets one); the resolved `hf.co/<repo>:<quant>` tag is written back into the model's dict as `m["tag"]`, exactly like the plain models, so every later step -- output folder naming, `ollama stop`, the config -- works the same for all three.

In [ ]:
sh("apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null")
if not shutil.which("ollama"):
    sh("curl -fsSL https://ollama.com/install.sh | sh > /dev/null")

OLLAMA_LOG = RUN_DIR / "ollama.log"
if subprocess.run(["curl", "-sf", "http://localhost:11434/api/tags"], capture_output=True).returncode:
    ollama = subprocess.Popen(["ollama", "serve"], stdout=open(OLLAMA_LOG, "a"), stderr=subprocess.STDOUT,
                              env={**os.environ, "OLLAMA_MAX_LOADED_MODELS": "1", "OLLAMA_FLASH_ATTENTION": "1",
                                   "OLLAMA_KV_CACHE_TYPE": "q8_0"})
    for _ in range(60):
        if not subprocess.run(["curl", "-sf", "http://localhost:11434/api/tags"], capture_output=True).returncode:
            break
        time.sleep(2)

import re

PREFERRED_QUANTS = ["Q4_K_M", "Q5_K_M", "IQ4_XS", "Q4_K_S", "Q5_K_S", "Q6_K"]

def pull_gguf(repo, quant):
    """Pulls `repo`'s GGUF file at `quant`, or the first of PREFERRED_QUANTS it actually has; returns
    the resolved Ollama tag."""
    try:
        files = [s["rfilename"] for s in json.load(urllib.request.urlopen(
            f"https://huggingface.co/api/models/{repo}", timeout=30))["siblings"]]
        print("GGUF files in the repo:", [f for f in files if f.endswith(".gguf")])
    except Exception as exc:
        files = []
        print("could not list the repo's files, trying quantizations in order:", exc)
    candidates = [quant] if quant else (
        [q for q in PREFERRED_QUANTS if any(re.search(rf"[-.]{q}\.gguf$", f, re.I) for f in files)] or PREFERRED_QUANTS)
    for cand in candidates:
        tag = f"hf.co/{repo}:{cand}"
        print("pulling", tag)
        if subprocess.run(["ollama", "pull", tag]).returncode == 0:
            return tag
    raise RuntimeError(f"none of {candidates} could be pulled from {repo}")

for m in MODELS:
    if "repo" in m:
        m["tag"] = pull_gguf(m["repo"], m.get("quant"))
        print(f"{m['repo']} -> {m['tag']}")
    else:
        sh(f"ollama pull {m['tag']}")
sh("ollama list")


## 3. The bulk legal corpus (attached input) and the eval set (from the repo)

`legal_txt/Evals/israeli_legal_eval.zip` is committed in the repo -- no separate dataset needed for it. The corpus vectordb is too big for git, so it comes from the dataset attached in step 1. Also checks every law and section the gold answers cite against the installed corpus (`corpus_coverage.md`) -- a missing law shows up there before it shows up as a wrong answer.

In [ ]:
# The corpus database: category folders (laws/, procedural_rules/) that hold chroma.sqlite3, found
# anywhere under /kaggle/input -- unpacked (Kaggle unzips uploads) or still zipped, at any folder depth.
# Folders of .jsonl records (the build's legal_corpus_jsonl) have the same category names but no
# chroma.sqlite3: they are ignored. The 28 Sept run copied that folder instead and ran without a corpus.
import zipfile
INPUT = pathlib.Path("/kaggle/input")
VECTORDB = pathlib.Path("data/legal_corpus_vectordb")
CATEGORIES_WANTED = ("laws", "procedural_rules")
db_roots = sorted({p.parent.parent for p in INPUT.rglob("chroma.sqlite3") if p.parent.name in CATEGORIES_WANTED})
print("database folders found:", [str(r) for r in db_roots] or "none")
shutil.rmtree(VECTORDB, ignore_errors=True)
VECTORDB.mkdir(parents=True)
if db_roots:
    for item in db_roots[0].iterdir():
        (shutil.copytree if item.is_dir() else shutil.copy2)(item, VECTORDB / item.name)
    print("copied", db_roots[0], "->", VECTORDB)
else:
    for z in sorted(INPUT.rglob("*.zip")):
        with zipfile.ZipFile(z) as archive:
            dbs = [n for n in archive.namelist()
                   if n.endswith("chroma.sqlite3") and len(n.split("/")) >= 2 and n.split("/")[-2] in CATEGORIES_WANTED]
            if not dbs:
                continue
            parts = dbs[0].split("/")  # [..., <category>, "chroma.sqlite3"]
            prefix = "/".join(parts[:-2]) + "/" if len(parts) > 2 else ""
            for member in archive.infolist():
                name = member.filename
                if name.startswith(prefix) and not name.endswith("/") and ".." not in name:
                    dest = VECTORDB / name[len(prefix):]
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    with archive.open(member) as fsrc, open(dest, "wb") as fdst:
                        shutil.copyfileobj(fsrc, fdst)
            print("unpacked", z, "->", VECTORDB)
            break

CORPUS_CATEGORIES = sorted(c for c in CATEGORIES_WANTED if (VECTORDB / c / "chroma.sqlite3").exists())
assert CORPUS_CATEGORIES, ("NO CORPUS DATABASE FOUND under /kaggle/input -- stopping instead of answering without it. "
                           "Attach the dataset with legal_corpus_vectordb (its laws/ and procedural_rules/ folders "
                           "must contain chroma.sqlite3).")
missing = [c for c in CATEGORIES_WANTED if c not in CORPUS_CATEGORIES]
if missing:
    print(f"[warn] the corpus has no {missing}: questions on those will retrieve nothing from them")

# Every collection must open and hold the chunks the build recorded.
BUILD_INFO = json.loads((VECTORDB / "_build_info.json").read_text()) if (VECTORDB / "_build_info.json").exists() else {}
import chromadb
bad = []
for c in CORPUS_CATEGORIES:
    n = chromadb.PersistentClient(path=str(VECTORDB / c)).get_collection(f"israeli_law_corpus_{c}").count()
    expected = BUILD_INFO.get("categories", {}).get(c, {}).get("chunks")
    print(f"{c:18s} {n:>9,} chunks" + (f" (build: {expected:,})" if expected else ""))
    if n == 0 or (expected and n != expected):
        bad.append(c)
if BUILD_INFO.get("chromadb_version") and BUILD_INFO["chromadb_version"] != chromadb.__version__:
    print(f"[note] built with chromadb {BUILD_INFO['chromadb_version']}, this machine has {chromadb.__version__}")
assert not bad, f"empty or incomplete collections {bad}: re-upload or rebuild the corpus before a long run"
print("corpus OK:", CORPUS_CATEGORIES)

EVAL_DIR = pathlib.Path("legal_txt/Evals")
import zipfile
zipfile.ZipFile(EVAL_DIR / "israeli_legal_eval.zip").extractall(EVAL_DIR)
EVAL = EVAL_DIR / "israeli_legal_eval"
assert (EVAL / "questions.jsonl").exists(), f"eval set not found under {EVAL}"

sh(f'python scripts/legal_data/check_corpus_coverage.py --gold {EVAL / "gold.jsonl"} --categories {",".join(CORPUS_CATEGORIES)} '
   f'--out "{RUN_DIR / "corpus_coverage.md"}" --json-out "{RUN_DIR / "corpus_coverage.json"}"', check=False)


## 4. The same 100 questions as the other runs

A proportional slice of each of the 9 categories, picked with `SAMPLE_SEED`.

In [ ]:
import random
from collections import Counter, defaultdict

def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

all_questions = load_jsonl(EVAL / "questions.jsonl")
QUESTIONS = RUN_DIR / "questions_sample.jsonl"
if LIMIT and LIMIT < len(all_questions):
    by_cat = defaultdict(list)
    for q in all_questions:
        by_cat[q["category"]].append(q)
    share, rng, sample = LIMIT / len(all_questions), random.Random(SAMPLE_SEED), []
    for qs in by_cat.values():
        k = max(1, round(len(qs) * share))
        picked = qs[:k] if SAMPLE_SEED is None else rng.sample(qs, k)
        sample += sorted(picked, key=lambda q: q["id"])
else:
    sample = all_questions
with open(QUESTIONS, "w", encoding="utf-8") as f:
    for q in sample:
        f.write(json.dumps(q, ensure_ascii=False) + "\n")
print(f"{len(sample)} questions ->", QUESTIONS)
print("by category:", dict(Counter(q["category"] for q in sample).most_common()))


## 4b. Resume: bring in an earlier run's results (if `RESUME_FROM` is set)

Copies the earlier `answers_*` files into place before anything runs, minus answers that came back empty or errored -- so those get redone instead of being skipped as "already done".

In [ ]:
def folder_name(tag):  # the same name step 5's model_dir gives a model's folder ("hf.co/x/y:Q4" -> "hf.co_x_y_Q4")
    return tag.replace(":", "_").replace("/", "_")

def find_eval_dir(root):
    hits = sorted(p for p in pathlib.Path(root).rglob(RUN_NAME) if p.is_dir() and (p / folder_name(MODELS[0]["tag"])).exists())
    return hits[0] if hits else None

if RESUME_FROM:
    src = RESUME_FROM
    if src == "auto":
        zips = sorted(pathlib.Path("/kaggle/input").rglob(f"*{RUN_NAME}*results*.zip"))
        assert len(zips) <= 1, f"several earlier results attached, remove all but one from Input: {zips}"
        src = zips[0] if zips else "/kaggle/input"
    src = pathlib.Path(src)
    if src.is_file() and src.suffix == ".zip":
        extract_to = pathlib.Path("/kaggle/working/resume_from")
        shutil.rmtree(extract_to, ignore_errors=True)
        zipfile.ZipFile(src).extractall(extract_to)
        src = extract_to
    prev = find_eval_dir(src)
    assert prev, f"no {RUN_NAME}/<model> folders found under {src}"
    for m in MODELS:
        prev_dir, out_dir = prev / folder_name(m["tag"]), RUN_DIR / folder_name(m["tag"])
        for rel in ("bulk500/answers_rag.jsonl", "cases/answers_rag.jsonl"):
            src_file = prev_dir / rel
            if src_file.exists():
                (out_dir / rel).parent.mkdir(parents=True, exist_ok=True)
                shutil.copy(src_file, out_dir / rel)
        prev_trace, out_trace = prev_dir / "llm_trace", out_dir / "llm_trace"
        if prev_trace.exists():
            out_trace.mkdir(parents=True, exist_ok=True)
            for f in prev_trace.glob("*.jsonl"):
                shutil.copy(f, out_trace / f.name)
    print("resumed from", prev)


## 5. Answer: each model in turn (resumable)

Each model gets its own config (Ollama, its tag, the 16k context, the timeout) and its own output folder and trace. Retrieval runs on the CPU (`CUDA_VISIBLE_DEVICES=""` for these scripts only; Ollama keeps the GPU). Each line of output is `id (category): Ns`.

In [ ]:
import yaml

base_cfg = yaml.safe_load(open("config/config.yaml", encoding="utf-8"))

def model_dir(tag):
    return RUN_DIR / tag.replace(":", "_").replace("/", "_")

def model_env(m):
    out = model_dir(m["tag"])
    for sub in ("bulk500", "cases", "llm_trace"):
        (out / sub).mkdir(parents=True, exist_ok=True)
    cfg = json.loads(json.dumps(base_cfg))
    orch = cfg["legal"]["orchestrator"]
    orch.update(backend="ollama", base_url="http://localhost:11434", model=m["tag"],
                max_model_len=CONTEXT_LENGTH, request_timeout_s=REQUEST_TIMEOUT_S)
    cfg.setdefault("logging", {})["llm_trace_dir"] = str(out / "llm_trace")
    cfg_path = out / "config.yaml"
    cfg_path.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding="utf-8")
    return out, {**os.environ, "DOCSLIDES_CONFIG": str(cfg_path), "CUDA_VISIBLE_DEVICES": "",
                 "DOCSLIDES_LEGAL_ORCHESTRATOR_BACKEND": "ollama",
                 "DOCSLIDES_LEGAL_ORCHESTRATOR_BASE_URL": "http://localhost:11434",
                 "DOCSLIDES_LEGAL_ORCHESTRATOR_MODEL": m["tag"],
                 "DOCSLIDES_LEGAL_ORCHESTRATOR_MAX_MODEL_LEN": str(CONTEXT_LENGTH),
                 "DOCSLIDES_LLM_TRACE_DIR": str(out / "llm_trace")}

def drop_failed(path):
    """Resume: remove answers that errored or came back empty, so this run redoes them."""
    if not path.exists():
        return 0
    rows = load_jsonl(path)
    good = [r for r in rows if not r.get("error") and str(r.get("answer") or r.get("work_file") or "").strip()]
    if len(good) < len(rows):
        with open(path, "w", encoding="utf-8") as f:
            for r in good:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return len(rows) - len(good)

for m in MODELS:
    out, env = model_env(m)
    log = out / "run.log"
    no_think = "" if m["thinking"] else " --no-thinking"
    answers = out / "bulk500" / "answers_rag.jsonl"
    print(f"\n{'=' * 20} {m['tag']}: questions {'=' * 20}  (redoing {drop_failed(answers)} failed)")
    sh(f'python scripts/legal_data/eval_run.py answer --questions "{QUESTIONS}" --top-k {TOP_K} '
       f'--max-tokens {ANSWER_MAX_TOKENS} --out "{answers}"{no_think}', log=log, env=env)
    if RUN_CASES:
        cases = out / "cases" / "answers_rag.jsonl"
        print(f"\n{'=' * 20} {m['tag']}: {CASE_LIMIT} cases {'=' * 20}  (redoing {drop_failed(cases)} failed)")
        sh(f'python scripts/legal_data/eval_cases.py answer --dir {EVAL} --top-k {CASE_TOP_K} '
           f'--max-tokens {CASE_MAX_TOKENS} --limit {CASE_LIMIT} --out "{cases}"{no_think}', log=log, env=env)
    subprocess.run(["ollama", "stop", m["tag"]], capture_output=True)  # free the GPU for the next model


## 6. Reasoning logs, judge-free scores, summary

- `reasoning_report.md`: per question/case, every call's reasoning, output, stop reason and timing;
  `reasoning_report_with_prompts.md` adds the full prompts (retrieved evidence included).
- `judge_free_report.*`: score.py (unmodified) without a judge -- the 16 multiple-choice and 18 yes/no
  label scores only; the judged part waits for review.

In [ ]:
import re, statistics

FOREIGN = re.compile(r"[\u0400-\u04FF\u0600-\u06FF\u4E00-\u9FFF\u3040-\u30FF\uAC00-\uD7A3\u0E00-\u0E7F]+|[A-Za-z]{4,}")
summary = {"run": RUN_NAME, "questions": len(sample), "cases": CASE_LIMIT if RUN_CASES else 0, "models": {}}

for m in MODELS:
    out = model_dir(m["tag"])
    traces = sorted((out / "llm_trace").glob("*.jsonl"))
    if traces:
        files = " ".join(f'"{p}"' for p in traces)
        sh(f'python scripts/llm_trace_report.py {files} > "{out / "reasoning_report.md"}"', check=False)
        sh(f'python scripts/llm_trace_report.py {files} --prompts > "{out / "reasoning_report_with_prompts.md"}"', check=False)
    answers = out / "bulk500" / "answers_rag.jsonl"
    judge_free = {}
    if answers.exists():
        sh(f'python {EVAL / "score.py"} report --gold {EVAL / "gold.jsonl"} --answers "{answers}" '
           f'--json-out "{out / "judge_free_report.json"}" > "{out / "judge_free_report.txt"}"', check=False)
        if (out / "judge_free_report.json").exists():
            rows = json.load(open(out / "judge_free_report.json", encoding="utf-8"))["rows"]
            mcq = [r["score"] for r in rows if r["category"] == "mcq_bar"]
            labels = [r["label_correct"] for r in rows if "label_correct" in r]
            judge_free = {"multiple_choice": round(statistics.mean(mcq), 3) if mcq else None, "mcq_n": len(mcq),
                          "yes_no_label": round(statistics.mean(labels), 3) if labels else None, "yes_no_n": len(labels)}
    ans = load_jsonl(answers) if answers.exists() else []
    case_rows = load_jsonl(out / "cases" / "answers_rag.jsonl") if (out / "cases" / "answers_rag.jsonl").exists() else []
    calls = [json.loads(l) for p in traces for l in open(p, encoding="utf-8") if l.strip()]
    by_site = defaultdict(list)
    for c in calls:
        by_site[c.get("call_site")].append(c)
    summary["models"][m["tag"]] = {
        "answers": len(ans), "errors": sum(bool(a.get("error")) for a in ans),
        "empty": sum(not str(a.get("answer") or "").strip() for a in ans),
        "repairs": dict(Counter(r for a in ans for r in a.get("repairs", []))),
        "answers_with_other_script_words": sum(bool(FOREIGN.search(a.get("answer") or "")) for a in ans),
        "cases": len(case_rows), "case_errors": sum(bool(c.get("error")) for c in case_rows),
        "judge_free": judge_free,
        "calls": {site: {"n": len(cs), "stopped_at_length": sum(c.get("done_reason") == "length" for c in cs),
                         "errors": sum(bool(c.get("error")) for c in cs),
                         "median_s": round(statistics.median([c.get("seconds") or 0 for c in cs]), 1),
                         "max_prompt_tokens": max((c.get("prompt_tokens") or 0) for c in cs),
                         "max_completion_tokens": max((c.get("completion_tokens") or 0) for c in cs)}
                  for site, cs in sorted(by_site.items())},
    }

(RUN_DIR / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1), encoding="utf-8")
lines = [f"# Kaggle eval run {RUN_NAME}", "", f"{summary['questions']} questions, {summary['cases']} cases", ""]
for tag, s in summary["models"].items():
    jf = s["judge_free"]
    lines += [f"## {tag}", "",
              f"- answers {s['answers']} (errors {s['errors']}, empty {s['empty']}), cases {s['cases']} (errors {s['case_errors']})",
              f"- judge-free: multiple choice {jf.get('multiple_choice')} (n={jf.get('mcq_n')}), yes/no label {jf.get('yes_no_label')} (n={jf.get('yes_no_n')})",
              f"- repairs {s['repairs']}; answers with other-script words: {s['answers_with_other_script_words']}", "",
              "| call site | calls | stopped at length | errors | median s | max prompt tok | max output tok |",
              "|---|---|---|---|---|---|---|"]
    lines += [f"| {site} | {c['n']} | {c['stopped_at_length']} | {c['errors']} | {c['median_s']} | {c['max_prompt_tokens']} | {c['max_completion_tokens']} |"
              for site, c in s["calls"].items()] + [""]
(RUN_DIR / "summary.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))


## 7. Environment record and the results zip (for Output)

In [ ]:
env_lines = []
for label, cmd in [("git", "git log -1 --format='%H %s'"), ("gpu", "nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv"),
                   ("ollama", "ollama --version"), ("models", "ollama list"),
                   ("python", "python -c \"import sys, chromadb, sentence_transformers, torch; print(sys.version); print('chromadb', chromadb.__version__, '| sentence-transformers', sentence_transformers.__version__, '| torch', torch.__version__)\"")]:
    res = subprocess.run(["bash", "-c", cmd], capture_output=True, text=True)
    env_lines += [f"## {label}", (res.stdout + res.stderr).strip(), ""]
env_lines += ["## settings", json.dumps({k: globals()[k] for k in ("BRANCH", "MODELS", "LIMIT", "SAMPLE_SEED", "RUN_CASES", "CASE_LIMIT",
              "TOP_K", "CASE_TOP_K", "CONTEXT_LENGTH", "ANSWER_MAX_TOKENS", "CASE_MAX_TOKENS", "REQUEST_TIMEOUT_S")}, indent=1)]
(RUN_DIR / "environment.txt").write_text("\n".join(env_lines), encoding="utf-8")

DEST = f"/kaggle/working/{RUN_NAME}_results.zip"
sh(f'cd /kaggle/working && zip -qr {DEST} AI-IZ/{RUN_DIR}')
print("results zip:", DEST, f"({pathlib.Path(DEST).stat().st_size / 1e6:.1f} MB)")
print("Send this zip back for the judging and the reasoning review.")
